In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

In [2]:
names = pd.read_csv("dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2931087,2,LUCIENNE,1914,25,32
1918063,2,ANNE-LISE,1986,85,3
1824552,2,ALYCIA,2003,22,3
792389,1,JEAN-CLAUDE,1941,27,71
1045866,1,LUCAS,2002,971,40


In [3]:
depts = gpd.read_file('departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
84,84,Vaucluse,"MULTIPOLYGON (((4.89291 44.36482, 4.90663 44.3..."
24,26,Drôme,"POLYGON ((4.80049 45.29836, 4.8588 45.30895, 4..."
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ..."
78,78,Yvelines,"POLYGON ((2.20059 48.90868, 2.16838 48.89508, ..."
66,66,Pyrénées-Orientales,"POLYGON ((2.16605 42.66392, 2.1762 42.65251, 2..."


In [4]:
# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
132707,52,Haute-Marne,"POLYGON ((4.67018 48.53189, 4.71801 48.54199, ...",1,ANTOINE,1999,52,10
2991060,69,Rhône,"POLYGON ((4.38808 46.21979, 4.39205 46.26302, ...",2,MARIA,1997,69,3
965164,50,Manche,"POLYGON ((-1.11962 49.35557, -1.11346 49.32795...",1,LÉON,1911,50,64
38255,66,Pyrénées-Orientales,"POLYGON ((2.16605 42.66392, 2.1762 42.65251, 2...",1,ALAIN,1941,66,14
3377238,61,Orne,"POLYGON ((-0.84094 48.75222, -0.81927 48.75413...",2,RAYMONDE,1917,61,16


In [5]:
#objectif 1 : chart des 20 prénoms les plus populaires de 2004
"""
annais : filtrer sur 2004
puis garder que les 20 premiers de nombre
puis faire le chart
"""

'\nannais : filtrer sur 2004\npuis garder que les 20 premiers de nombre\npuis faire le chart\n'

In [6]:
names_2004 = just_names[just_names["annais"]=="2004"]
names_2004.head(10)

,sexe,preusuel,annais,dpt,nombre
10987,1,AARON,2004,02,4
10988,1,AARON,2004,03,3
10989,1,AARON,2004,08,4
10990,1,AARON,2004,12,3
10991,1,AARON,2004,13,11
10992,1,AARON,2004,14,4
10993,1,AARON,2004,21,3
10994,1,AARON,2004,25,4
10995,1,AARON,2004,29,4
10996,1,AARON,2004,31,8


In [7]:
grouped_2004 = names_2004.groupby(['preusuel', 'sexe'], as_index=False).sum(numeric_only=True)

grouped_2004


,preusuel,sexe,nombre
0,AALIYAH,2,29
1,AARON,1,281
2,ABD,1,3
3,ABDALLAH,1,82
4,ABDEL,1,22
...,...,...,...
3633,ZOHRA,2,18
3634,ZORAN,1,3
3635,ZOUMANA,1,3
3636,ZOÉ,2,2280


In [8]:
ordered_2004 = grouped_2004.sort_values(by=["nombre"], ascending=False)

In [9]:
top_20 = ordered_2004.iloc[0:20]

In [10]:
base = alt.Chart(top_20).mark_bar().encode(
    x = 'nombre:Q',
    y =alt.Y('preusuel:N', sort='-x')
)

base

alt.Chart(...)

In [11]:
slider = alt.binding_range(min=1900, max=2020, step=1, name='year:')
year = alt.param(value=2004, bind=slider)
base.add_params(year)

alt.Chart(...)

In [12]:
def table(year): 
    names_2004 = just_names[just_names["annais"]==str(year)]
    grouped_2004 = names_2004.groupby(['preusuel', 'sexe'], as_index=False).sum(numeric_only=True)
    ordered_2004 = grouped_2004.sort_values(by=["nombre"], ascending=False)
    top_20 = ordered_2004.iloc[0:20]
    return top_20


In [13]:
just_names

,sexe,preusuel,annais,dpt,nombre
10885,1,AADIL,1983,84,3
10886,1,AADIL,1992,92,3
10888,1,AAHIL,2016,95,3
10892,1,AARON,1962,75,3
10893,1,AARON,1976,75,3
...,...,...,...,...,...
3727545,2,ZYA,2013,44,4
3727546,2,ZYA,2013,59,3
3727547,2,ZYA,2017,974,3
3727548,2,ZYA,2018,59,3


In [141]:
just_names_year = just_names.groupby(['annais', 'preusuel', 'sexe'], as_index=False).sum(numeric_only=True)
# just_names_ordered = just_names_year.sort_values(by=["nombre"], ascending=False)


just_names_year.head(20)



,annais,preusuel,sexe,nombre
0,1900,ABEL,1,382
1,1900,ABRAHAM,1,9
2,1900,ACHILLE,1,152
3,1900,ACHILLES,1,4
4,1900,ADAM,1,9
5,1900,ADELAIDE,2,143
6,1900,ADELHEID,2,3
7,1900,ADELINA,2,27
8,1900,ADELINE,2,169
9,1900,ADOLPHE,1,464


In [142]:
names = just_names_year

In [16]:
# # The spacing will only show up in your IDE, not on this doc page
# options = ['Europe', 'Japan', 'USA']
# labels = [option + ' ' for option in options]

# input_dropdown = alt.binding_radio(
#     # Add the empty selection which shows all when clicked
#     options=options + [None],
#     labels=labels + ['All'],
#     name='Region: '
# )
# selection = alt.selection_point(
#     fields=['Origin'],
#     bind=input_dropdown,
# )

# alt.Chart(cars).mark_point().encode(
#     x='Horsepower:Q',
#     y='Miles_per_Gallon:Q',
#     # We need to set a constant domain to preserve the colors
#     # when only one region is shown at a time
#     color=alt.Color('Origin:N').scale(domain=options),
# ).add_params(
#     selection
# ).transform_filter(
    # selection
# ) 

In [143]:
names["annees"] = names["annais"].apply(int)

In [18]:
names.head(10
)

,annais,preusuel,sexe,nombre,annees
0,1900,ABEL,1,382,1900
1,1900,ABRAHAM,1,9,1900
2,1900,ACHILLE,1,152,1900
3,1900,ACHILLES,1,4,1900
4,1900,ADAM,1,9,1900
5,1900,ADELAIDE,2,143,1900
6,1900,ADELHEID,2,3,1900
7,1900,ADELINA,2,27,1900
8,1900,ADELINE,2,169,1900
9,1900,ADOLPHE,1,464,1900


In [19]:
from IPython.display import display, HTML

# 1. Injection du CSS pour modifier l'apparence visuelle globale du slider
display(HTML("""
<style>
    .vega-bind {
        font-size: 18px !important;       /* Augmente la taille du texte de l'étiquette */
        font-family: 'Segoe UI', sans-serif;
        font-weight: 500;
        # color: #333;
        margin-top: 20px;                 /* Espace entre le graphique et le slider */
        margin-bottom: 20px;
    }

    /* Modifier la barre horizontale du slider */
    .vega-bind input[type="range"] {
        width: 500px !important;          /* LARGEUR VISUELLE DU SLIDER (par défaut ~150px) */
        height: 10px;                     /* Épaisseur de la barre du curseur */
        # background: #e0e0e0;
        border-radius: 5px;
        appearance: none;
        cursor: pointer;
        margin-left: 15px;                /* Espace entre le texte et la barre */
        margin-right: 15px;
    }

    /* Modifier le bouton coulissant (la "puce" ou "le bouton rond") du slider */
    .vega-bind input[type="range"]::-webkit-slider-thumb {
        appearance: none;
        width: 20px;                      /* Largeur de la puce */
        height: 20px;                     /* Hauteur de la puce */
        border-radius: 50%;               /* Rend le bouton rond */
        background: #4A90E2;              /* Couleur du bouton */
        cursor: pointer;
    }
</style>
"""))

In [20]:
slider = alt.binding_range(min=1900, max=2020, step=1, name='year:')
year = alt.param(value=2004, bind=slider)
selection = alt.selection_point(fields=["annees"], bind=slider)


# options = [(i) for i in range(1900, 2021)]
# labels = [str(option) + ' ' for option in options]

# input_dropdown = alt.binding_radio(
#     # Add the empty selection which shows all when clicked
#     options=options + [None],
#     labels=[str(i) for i in labels] + ['All'],
#     name='Region: '
# )
# selection = alt.selection_point(
#     fields=['annees'],
#     bind=input_dropdown,
# )


base = alt.Chart(names).mark_bar().encode(
    x = 'nombre:Q',
    y =alt.Y('preusuel:N', sort='-x'),

    
).add_params(
    year
).transform_filter(
    alt.datum.annais == alt.expr.toString(year),
).transform_window(    
    rank ='rank(nombre)',
    sort=[alt.SortField('nombre', order='descending')]
).transform_filter(
    alt.datum.rank <=20
).properties(
    width=600,
    height=400
)

base



alt.Chart(...)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def ajouter_couleurs_etendues(df_input):
    df = df_input.copy()
    
    df['sexe'] = df['sexe'].astype(str).str.strip()
    df['preusuel'] = df['preusuel'].astype(str).str.strip()
    
    valeurs_sexes = df['sexe'].unique()
    valeur_garcon = "1" if "1" in valeurs_sexes else "M"
    valeur_fille = "2" if "2" in valeurs_sexes else "F"
    
    unique_boys = df[df['sexe'] == valeur_garcon]['preusuel'].unique()
    unique_girls = df[df['sexe'] == valeur_fille]['preusuel'].unique()
    
    cmap_hommes = mcolors.LinearSegmentedColormap.from_list(
        "BleuVert", ["#1e3c72", "#2a5298", "#00c6ff", "#0072ff", "#11998e", "#38ef7d"]
    )
    
    cmap_femmes = mcolors.LinearSegmentedColormap.from_list(
        "VioletRose", ["#4e54c8", "#b92b27", "#f857a6", "#ff5858", "#ee0979", "#ff6a00"]
    )
    
    dict_couleurs = {}
    
    nb_g = len(unique_boys)
    for i, prenom in enumerate(unique_boys):
        position = i / max(1, nb_g - 1)
        dict_couleurs[(prenom, valeur_garcon)] = mcolors.to_hex(cmap_hommes(position))
        
    nb_f = len(unique_girls)
    for i, prenom in enumerate(unique_girls):
        position = i / max(1, nb_f - 1)
        dict_couleurs[(prenom, valeur_fille)] = mcolors.to_hex(cmap_femmes(position))
        
    df['couleur'] = df.apply(lambda row: dict_couleurs.get((row['preusuel'], row['sexe']), "#888888"), axis=1)
    
    return df

In [30]:
import colorsys

def generer_couleurs_eloignees(n, clarte=0.5, saturation=0.8):
    """
    Génère n couleurs spectralement distantes.
    
    :param n: Nombre de couleurs souhaitées.
    :param clarte: Une valeur unique (ex: 0.5) ou une liste/tuple de valeurs (ex: [0.4, 0.7]) 
                   pour faire varier la luminosité.
    :param saturation: Saturation des couleurs (entre 0.0 et 1.0).
    :return: Liste de codes couleur Hexadécimaux.
    """
    couleurs = []
    
    for i in range(n):
        # Division égale du cercle chromatique (Teinte entre 0.0 et 1.0)
        teinte = i / n
        
        # Gestion de la clarté (fixe ou alternée)
        if isinstance(clarte, (list, tuple)):
            c = clarte[i % len(clarte)]  # Alterne entre les clartés fournies
        else:
            c = clarte
            
        # Conversion HSL (HLS en Python) vers RGB (valeurs entre 0.0 et 1.0)
        r, g, b = colorsys.hls_to_rgb(teinte, c, saturation)
        
        # Conversion en format Hexadécimal (#RRGGBB)
        hex_color = f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"
        couleurs.append(hex_color)
        
    return couleurs

In [38]:
def make_lst_couleurs(df_input):
    df = df_input.copy()

    prenoms_fille = df[df["sexe"]==2]["preusuel"].unique().tolist()
    prenoms_garcon = df[df["sexe"]==1]["preusuel"].unique().tolist()
    print(prenoms_fille)
    print(prenoms_garcon)
    prenoms = prenoms_fille + prenoms_garcon
    n = len(prenoms)
    lst_couleurs = generer_couleurs_eloignees(n, [0.3, 0.6, 0.9], 0.8)
    
    return lst_couleurs



def make_dico_couleurs(prenoms, prenoms_fille, prenoms_garcon, lst_couleurs): 
    dico = {}
    for i in range(len(prenoms)): 
        if i<len(prenoms_fille):
            dico[prenoms_fille[i], 2] = lst_couleurs[i]
        else : 
            dico[prenoms_garcon[i - len(prenoms_fille)], 1] = lst_couleurs[i]
    return dico
    


In [39]:
lst_couleurs = make_lst_couleurs(names)

['ADELAIDE', 'ADELHEID', 'ADELINA', 'ADELINE', 'ADOLPHINE', 'ADRIENNE', 'ADÈLE', 'AGATHE', 'AGLAE', 'AGNÈS', 'AIDA', 'AIMÉE', 'ALBANIE', 'ALBERTA', 'ALBERTE', 'ALBERTINE', 'ALBINE', 'ALEXANDRINE', 'ALEXIA', 'ALEXINA', 'ALEXINE', 'ALFREDA', 'ALFREDINE', 'ALICE', 'ALIDA', 'ALINE', 'ALIX', 'ALIXE', 'ALLIETTE', 'ALPHENA', 'ALPHONSINE', 'ALZIRE', 'AMALIE', 'AMANDA', 'AMANDINE', 'AMBROISINE', 'AMÉLIA', 'AMÉLIE', 'ANAISE', 'ANASTASIE', 'ANAÏS', 'ANDREANNE', 'ANDRÉ', 'ANDRÉA', 'ANDRÉE', 'ANGE', 'ANGELA', 'ANGELINA', 'ANGELINE', 'ANGELIQUE', 'ANGÈLE', 'ANITA', 'ANNA', 'ANNE', 'ANNE-MARIE', 'ANNETTE', 'ANNITA', 'ANNONCIA', 'ANNONCIADE', 'ANTHELMETTE', 'ANTHELMINE', 'ANTOINETTE', 'ANTOINISE', 'ANTONIA', 'ANTONIE', 'ANTONINE', 'APOLLINE', 'APOLLONIE', 'APPOLINE', 'APPOLONIE', 'ARGENTINE', 'ARLETTE', 'ARMANCE', 'ARMANDE', 'ARMANDINE', 'ARMANTINE', 'ARSENE', 'ARTHEMISE', 'ASSOMPTION', 'ASSUNTA', 'AUGUSTA', 'AUGUSTINA', 'AUGUSTINE', 'AURORE', 'AURÉLIE', 'AZELINE', 'AZEMA', 'BABETTE', 'BAPTISTINE', 'B

In [41]:
print(lst_couleurs)

['#890f0f', '#ea4747', '#f9d1d1', '#890f0f', '#ea4747', '#f9d1d1', '#890f0f', '#ea4747', '#f9d1d1', '#890f0f', '#ea4847', '#f9d1d1', '#890f0f', '#ea4847', '#f9d1d1', '#890f0f', '#ea4847', '#f9d1d1', '#89100f', '#ea4847', '#f9d1d1', '#89100f', '#ea4847', '#f9d1d1', '#89100f', '#ea4847', '#f9d1d1', '#89100f', '#ea4947', '#f9d1d1', '#89100f', '#ea4947', '#f9d1d1', '#89100f', '#ea4947', '#f9d1d1', '#89100f', '#ea4947', '#f9d1d1', '#89110f', '#ea4947', '#f9d1d1', '#89110f', '#ea4a47', '#f9d1d1', '#89110f', '#ea4a47', '#f9d1d1', '#89110f', '#ea4a47', '#f9d1d1', '#89110f', '#ea4a47', '#f9d1d1', '#89110f', '#ea4a47', '#f9d1d1', '#89110f', '#ea4a47', '#f9d1d1', '#89120f', '#ea4b47', '#f9d2d1', '#89120f', '#ea4b47', '#f9d2d1', '#89120f', '#ea4b47', '#f9d2d1', '#89120f', '#ea4b47', '#f9d2d1', '#89120f', '#ea4b47', '#f9d2d1', '#89120f', '#ea4c47', '#f9d2d1', '#89120f', '#ea4c47', '#f9d2d1', '#89120f', '#ea4c47', '#f9d2d1', '#89130f', '#ea4c47', '#f9d2d1', '#89130f', '#ea4c47', '#f9d2d1', '#89130f'

,annais,preusuel,sexe,nombre,annees
0,1900,ABEL,1,382,1900
1,1900,ABRAHAM,1,9,1900
2,1900,ACHILLE,1,152,1900
3,1900,ACHILLES,1,4,1900
4,1900,ADAM,1,9,1900
...,...,...,...,...,...
95,1900,ANNE,2,3440,1900
96,1900,ANNE-MARIE,2,62,1900
97,1900,ANNET,1,17,1900
98,1900,ANNETTE,2,181,1900


In [144]:
names_colores = ajouter_couleurs_etendues(names)

In [46]:
import colorsys

def generer_couleurs_eloignees(n, clarte=0.5, saturation=0.8):
    """
    Génère n couleurs spectralement distantes.
    
    :param n: Nombre de couleurs souhaitées.
    :param clarte: Une valeur unique (ex: 0.5) ou une liste/tuple de valeurs (ex: [0.4, 0.7]) 
                   pour faire varier la luminosité.
    :param saturation: Saturation des couleurs (entre 0.0 et 1.0).
    :return: Liste de codes couleur Hexadécimaux.
    """
    couleurs = []
    
    for i in range(n):
        # Division égale du cercle chromatique (Teinte entre 0.0 et 1.0)
        teinte = i / n
        
        # Gestion de la clarté (fixe ou alternée)
        if isinstance(clarte, (list, tuple)):
            c = clarte[i % len(clarte)]  # Alterne entre les clartés fournies
        else:
            c = clarte
            
        # Conversion HSL (HLS en Python) vers RGB (valeurs entre 0.0 et 1.0)
        r, g, b = colorsys.hls_to_rgb(teinte, c, saturation)
        
        # Conversion en format Hexadécimal (#RRGGBB)
        hex_color = f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"
        couleurs.append(hex_color)
        
    return couleurs

In [48]:
ma_palette = generer_couleurs_eloignees(n=6, clarte=[0.4, 0.65], saturation=0.85)

In [56]:
slider = alt.binding_range(min=1900, max=2020, step=1, name='year:')
year = alt.param(value=2004, bind=slider)
selection = alt.selection_point(fields=["annees"], bind=slider)

base = alt.Chart(names_colores).mark_bar().encode(
    x = 'nombre:Q',
    y =alt.Y('preusuel:N', sort='-x'),
# color=alt.Color('couleur:N').scale(
#     domain=names_colores['couleur'].unique().tolist(), 
#     range=names_colores['couleur'].unique().tolist()
# ).legend(None)

color=alt.Color(
        'preusuel:N', 
        scale=alt.Scale(range=ma_palette) # On applique notre palette ici
    )
    
).add_params(
    year
).transform_filter(
    alt.datum.annais == alt.expr.toString(year),
).transform_window(    
    rank ='rank(nombre)',
    sort=[alt.SortField('nombre', order='descending')]
).transform_filter(
    alt.datum.rank <=20
).properties(
    width=600,
    height=400
)

In [57]:
base

alt.Chart(...)

In [127]:
import colorsys

def generer_couleurs_par_groupe(n, hue_start, hue_end, clarte=[0.2, 0.4, 0.6, 0.8], saturation=0.8):
    """Génère n couleurs distantes limitées à une plage du cercle chromatique (0.0 à 1.0)."""
    couleurs = []
    if n == 0:
        return couleurs
        
    for i in range(n):
        # Répartition des teintes uniquement entre hue_start et hue_end
        if n == 1:
            teinte = (hue_start + hue_end) / 2
        else:
            teinte = hue_start + (i / (n - 1)) * (hue_end - hue_start)
        
        # Variation de la clarté
        c = clarte[i % len(clarte)] if isinstance(clarte, (list, tuple)) else clarte
        
        # Conversion Hexadécimale
        r, g, b = colorsys.hls_to_rgb(teinte, c, saturation)
        hex_color = f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"
        couleurs.append(hex_color)
        
    return couleurs

In [128]:
# import altair as alt
# import pandas as pd

# --- 1. Données d'exemple (avec un prénom mixte "Camille") ---
# data = pd.DataFrame({
#     'Prenom': ['Alice', 'Emma', 'Camille', 'Bob', 'Charlie', 'Camille'],
#     'Sexe': ['F', 'F', 'F', 'H', 'H', 'H'],
#     'Score': [20, 14, 18, 25, 17, 12]
# })

def add_couleur_df(data) : 
# Étape cruciale : On crée une colonne d'identification unique pour la couleur
    data['Id_Couleur'] = data['preusuel'] + " (" + data['sexe'].astype(str) + ")"

    # --- 2. Génération des palettes distinctes par Sexe ---
    # On extrait les identifiants uniques triés par genre
    identifiants_filles = sorted(data[data['sexe'] == 2]['Id_Couleur'].unique())
    identifiants_garcons = sorted(data[data['sexe'] == 1]['Id_Couleur'].unique())
    print(identifiants_filles)

    # Filles : Teintes chaudes (du Violet/Rose au Rouge -> 0.75 à 0.98)
    couleurs_filles = generer_couleurs_par_groupe(len(identifiants_filles), 0.5, 0.99)
    print(couleurs_filles)

    # Garçons : Teintes froides (du Vert au Bleu/Cyan -> 0.35 à 0.68)
    couleurs_garcons = generer_couleurs_par_groupe(len(identifiants_garcons), 0.035, 0.68)

    # --- 3. Création du dictionnaire de correspondance global ---
    mapping_couleurs = {}
    mapping_couleurs.update(zip(identifiants_filles, couleurs_filles))
    print(mapping_couleurs)
    mapping_couleurs.update(zip(identifiants_garcons, couleurs_garcons))

    return mapping_couleurs

In [146]:
mapping_couleurs = add_couleur_df(names)

['AALIYA (2)', 'AALIYAH (2)', 'AALYA (2)', 'AALYAH (2)', 'ABBIE (2)', 'ABBY (2)', 'ABBY-GAËLLE (2)', 'ABBYGAELLE (2)', 'ABBYGAËLLE (2)', 'ABDON (2)', 'ABDONIE (2)', 'ABDONISE (2)', 'ABEL (2)', 'ABELIA (2)', 'ABELINE (2)', 'ABELLA (2)', 'ABELLE (2)', 'ABI (2)', 'ABIBA (2)', 'ABIBATA (2)', 'ABIGAEL (2)', 'ABIGAELLE (2)', 'ABIGAIL (2)', 'ABIGAËL (2)', 'ABIGAËLLE (2)', 'ABIGAÏL (2)', 'ABINAYA (2)', 'ABIR (2)', 'ABIRA (2)', 'ABISHA (2)', 'ABLA (2)', 'ABRAR (2)', 'ABRIL (2)', 'ABSA (2)', 'ABY (2)', 'ABYGAEL (2)', 'ABYGAELLE (2)', 'ABYGAËL (2)', 'ABYGAËLLE (2)', 'ACACIA (2)', 'ACELYA (2)', 'ACHILLE (2)', 'ACIA (2)', 'ACIL (2)', 'ADA (2)', 'ADALINE (2)', 'ADAM (2)', 'ADAMA (2)', 'ADAME (2)', 'ADDA (2)', 'ADDISON (2)', 'ADELA (2)', 'ADELAIDE (2)', 'ADELAÏDE (2)', 'ADELE (2)', 'ADELHEID (2)', 'ADELIA (2)', 'ADELIE (2)', 'ADELINA (2)', 'ADELINE (2)', 'ADELPHINE (2)', 'ADELYA (2)', 'ADELYNE (2)', 'ADIA (2)', 'ADINA (2)', 'ADINE (2)', 'ADIXIA (2)', 'ADIZA (2)', 'ADJA (2)', 'ADJARATOU (2)', 'ADJIA (

In [147]:
names

,annais,preusuel,sexe,nombre,annees,Id_Couleur
0,1900,ABEL,1,382,1900,ABEL (1)
1,1900,ABRAHAM,1,9,1900,ABRAHAM (1)
2,1900,ACHILLE,1,152,1900,ACHILLE (1)
3,1900,ACHILLES,1,4,1900,ACHILLES (1)
4,1900,ADAM,1,9,1900,ADAM (1)
...,...,...,...,...,...,...
257341,2020,ÉVA,2,156,2020,ÉVA (2)
257342,2020,ÉVAN,1,62,2020,ÉVAN (1)
257343,2020,ÉZIO,1,12,2020,ÉZIO (1)
257344,2020,ÉZÉCHIEL,1,11,2020,ÉZÉCHIEL (1)


In [148]:
print(mapping_couleurs)
names['Couleur_Hex'] = names['Id_Couleur'].map(mapping_couleurs)

{'AALIYA (2)': '#0a5b5b', 'AALIYAH (2)': '#14b7b7', 'AALYA (2)': '#47eaea', 'AALYAH (2)': '#a3f4f4', 'ABBIE (2)': '#0a5b5b', 'ABBY (2)': '#14b7b7', 'ABBY-GAËLLE (2)': '#47eaea', 'ABBYGAELLE (2)': '#a3f4f4', 'ABBYGAËLLE (2)': '#0a5b5b', 'ABDON (2)': '#14b7b7', 'ABDONIE (2)': '#47eaea', 'ABDONISE (2)': '#a3f4f4', 'ABEL (2)': '#0a5b5b', 'ABELIA (2)': '#14b6b7', 'ABELINE (2)': '#47e9ea', 'ABELLA (2)': '#a3f4f4', 'ABELLE (2)': '#0a5b5b', 'ABI (2)': '#14b6b7', 'ABIBA (2)': '#47e9ea', 'ABIBATA (2)': '#a3f4f4', 'ABIGAEL (2)': '#0a5b5b', 'ABIGAELLE (2)': '#14b6b7', 'ABIGAIL (2)': '#47e9ea', 'ABIGAËL (2)': '#a3f4f4', 'ABIGAËLLE (2)': '#0a5b5b', 'ABIGAÏL (2)': '#14b6b7', 'ABINAYA (2)': '#47e9ea', 'ABIR (2)': '#a3f4f4', 'ABIRA (2)': '#0a5b5b', 'ABISHA (2)': '#14b6b7', 'ABLA (2)': '#47e8ea', 'ABRAR (2)': '#a3f3f4', 'ABRIL (2)': '#0a5a5b', 'ABSA (2)': '#14b5b7', 'ABY (2)': '#47e8ea', 'ABYGAEL (2)': '#a3f3f4', 'ABYGAELLE (2)': '#0a5a5b', 'ABYGAËL (2)': '#14b5b7', 'ABYGAËLLE (2)': '#47e8ea', 'ACACIA (

In [149]:
slider = alt.binding_range(min=1900, max=2020, step=1, name='year:')
year = alt.param(value=2004, bind=slider)
selection = alt.selection_point(fields=["annees"], bind=slider)

base = alt.Chart(names).mark_bar().encode(
    x = 'nombre:Q',
    y =alt.Y('preusuel:N', sort='-x'),
# color=alt.Color('couleur:N').scale(
#     domain=names_colores['couleur'].unique().tolist(), 
#     range=names_colores['couleur'].unique().tolist()
# ).legend(None)

# color=alt.Color(
#         'preusuel:N', 
#         scale=alt.Scale(range=ma_palette) # On applique notre palette ici
#     )

color=alt.Color(
        'Id_Couleur:N', # On colore basé sur notre identifiant unique
        # legend=alt.Legend(title="Prénoms (Sexe)"),
        # legend=None,
        scale=alt.Scale(range={"field": "Couleur_Hex"}
            # domain=list(mapping_couleurs.keys()),
            # range=list(mapping_couleurs.values())
        )    
)

# color=alt.Color(
#         'Couleur_Hex:N', 
#         scale=None # Dit à Altair : "Cette colonne contient déjà les vraies couleurs, utilise-les directement"
#     )

).add_params(
    year
).transform_filter(
    alt.datum.annais == alt.expr.toString(year),
).transform_window(    
    rank ='rank(nombre)',
    sort=[alt.SortField('nombre', order='descending')]
).transform_filter(
    alt.datum.rank <=20
).properties(
    width=600,
    height=400
)

base

alt.Chart(...)